In [1]:
# import the modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
import json

# load the json type data file
df = pd.read_json('simple.json')
df.head()

,id,brand,model,year,owned_by,acquired,price
0,1,Chrysler,Voyager,2001,Wolff-Trantow,28/12/2016,11565
1,2,Volkswagen,Jetta,1995,"Schuppe, Pfeffer and Klein",20/4/2016,52431
2,3,Porsche,Cayenne,2005,Mante Group,11/6/2020,75552
3,4,Porsche,928,1994,Wisozk Group,17/6/2019,30331
4,5,Mercedes-Benz,SL-Class,2007,Schiller-Littel,20/7/2018,62385


In [2]:
# Load the .xml file
df = pd.read_xml("simple.xml", xpath=".", parser="etree")
print(df)

   id       brand   model  year                    owned_by    acquired  price
0   1  Oldsmobile  Aurora  2003  Herzog, Rodriguez and Howe  21/10/2016  32571


In [3]:
# Load the data from an API
url = 'https://edu.frostbit.fi/api/events/en'
response = requests.get(url)

# Convert response to JSON
data = response.json()

# Convert JSON to Dataframe
df = pd.json_normalize(data)

df.head()

,name,date,categories,address.street_address,address.postal_code
0,Miss Blomcreutz's House Tour,6.7.2026,"[cultural events, guidance, maritime helsinki,...",Suomenlinna B 40,00190
1,Red Nose Company: Don Quixote (in English),13.11.2026,[theatre (art forms)],Aleksis Kiven katu 17/ Bruno Granholmin kuja,00510
2,"Korjaamo's Midsummer: FreeRap, The Moontwins +...",19.6.2026,[],Töölönkatu 51 a-b,00250
3,Short stories in English,14.9.2026,"[craft skills, cultural events, finnoo, langua...",Suomenlahdentie 1,02230
4,Drawing Workshop at Studio Aalto,29.8.2026,"[architecture, cultural events, drawings (work...",Tiilimäki 20,00330


In [4]:
# Check the columns
df.columns

Index(['name', 'date', 'categories', 'address.street_address',
       'address.postal_code'],
      dtype='object')

In [5]:
# Make each category a seperate row
df = df.explode('categories')
print(df)

                                                 name        date  \
0                       Miss Blomcreutz's House Tour     6.7.2026   
0                       Miss Blomcreutz's House Tour     6.7.2026   
0                       Miss Blomcreutz's House Tour     6.7.2026   
0                       Miss Blomcreutz's House Tour     6.7.2026   
1          Red Nose Company: Don Quixote (in English)  13.11.2026   
..                                                ...         ...   
11  Korjaamo's Midsummer: Marko Haavisto & Poutaha...   20.6.2026   
11  Korjaamo's Midsummer: Marko Haavisto & Poutaha...   20.6.2026   
12                           Beginning Art Exhibition   11.6.2026   
12                           Beginning Art Exhibition   11.6.2026   
12                           Beginning Art Exhibition   11.6.2026   

             categories                        address.street_address  \
0       cultural events                              Suomenlinna B 40   
0              guidance  

In [6]:
# define the URL of the webpage and download the raw HTML
url = "https://en.wikipedia.org/wiki/Rovaniemi"

# modern websites often require headers set correctly before 
# we can remotely read the website contents (HTML)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

# load the page data with headers
page = requests.get(url, headers=headers)
print(page.text)


<!DOCTYPE html>
<html lang="en">
<meta charset="utf-8">
<title>Wikimedia Error</title>
<style>
* { margin: 0; padding: 0; }
body { background: #fff; font: 15px/1.6 sans-serif; color: #333; }
.content { margin: 7% auto 0; padding: 2em 1em 1em; max-width: 640px; display: flex; flex-direction: row; flex-wrap: wrap; }
.footer { clear: both; margin-top: 14%; border-top: 1px solid #e5e5e5; background: #f9f9f9; padding: 2em 0; font-size: 0.8em; text-align: center; }
img { margin: 0 2em 2em 0; }
a img { border: 0; }
h1 { margin-top: 1em; font-size: 1.2em; }
.content-text { flex: 1; }
p { margin: 0.7em 0 1em 0; }
a { color: #0645ad; text-decoration: none; }
a:hover { text-decoration: underline; }
code { font-family: sans-serif; }
summary { font-weight: bold; cursor: pointer; }
details[open] { background: #970302; color: #dfdedd; }
.text-muted { color: #777; }
@media (prefers-color-scheme: dark) {
  a { color: #9e9eff; }
  body { background: transparent; color: #ddd; }
  .footer { border-top: 1p

In [7]:
# create a "soup object" of the new raw web page
soup = BeautifulSoup(page.text, "html.parser")

# The infobox is the table on the right side of the article
# Obtained from the Chatgpt
for row in soup.select("table.infobox tr"):
    if row.find("th") and "Coordinates" in row.find("th").get_text():
        coordinates = row.find("td").get_text(" ", strip=True)
        print(coordinates)

In [8]:
# Get the nicknames of Rovaniemi (under coat of arms)
# Obtained from the Chatgpt
for text in soup.find_all(string=lambda text: text and "Nicknames" in text):
    print(text)

In [9]:
# Get the Total Population
# Obtained from the Chatgpt
rows = soup.select("table.infobox tr")

for i, row in enumerate(rows):
    if row.find("th") and "Population" in row.find("th").get_text():
        population = rows[i + 1].find("td").get_text(" ", strip=True)
        print("Population:", population)

In [10]:
# Get the total area
# Obtained from the Chatgpt
for row in soup.select("table.infobox tr"):
    if "Total" in row.get_text():
        area = row.select_one("td.infobox-data").get_text(" ", strip=True)
        print(area)

In [11]:
# Convert this number into a float decimal
area = "8,016.75 km²"

area = area.replace(",", "")
area = area.replace("km²", "")
area = float(area)

print(area)

8016.75


In [12]:
# define the URL of the webpage and download the raw HTML
url = "https://en.wikipedia.org/wiki/List_of_countries_by_average_yearly_temperature"

# modern websites often require headers set correctly before 
# we can remotely read the website contents (HTML)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}


data = pd.read_html(url, storage_options=headers)
actual_data = data[0]
actual_data

,Unnamed: 0,Country or region,Continent,Temperature
0,1,Burkina Faso,Africa,30.40 °C (86.72 °F)
1,2,Mali,Africa,29.21 °C (84.58 °F)
2,3,Aruba,South America,29.17 °C (84.51 °F)
3,4,Senegal,Africa,28.90 °C (84.02 °F)
4,5,Mauritania,Africa,28.82 °C (83.88 °F)
...,...,...,...,...
230,231,Mongolia,Asia,1.07 °C (33.93 °F)
231,232,Russia,Asia and Europe,−3.79 °C (25.18 °F)
232,233,Canada,North America,−4.03 °C (24.75 °F)
233,234,Svalbard and Jan Mayen,Europe,−6.78 °C (19.80 °F)


In [13]:
# Change the name of the value column to "avg_temp"
actual_data = actual_data.rename(columns={"Temperature": "avg_temp"})
actual_data.head()

,Unnamed: 0,Country or region,Continent,avg_temp
0,1,Burkina Faso,Africa,30.40 °C (86.72 °F)
1,2,Mali,Africa,29.21 °C (84.58 °F)
2,3,Aruba,South America,29.17 °C (84.51 °F)
3,4,Senegal,Africa,28.90 °C (84.02 °F)
4,5,Mauritania,Africa,28.82 °C (83.88 °F)


In [14]:
#  Clean up the column so that only the numeric °C-value is present (no other characters)
actual_data["avg_temp"] = actual_data["avg_temp"].str.extract(r"(\d+\.\d+)").astype(float)
print(actual_data.head())

   Unnamed: 0 Country or region      Continent  avg_temp
0           1      Burkina Faso         Africa     30.40
1           2              Mali         Africa     29.21
2           3             Aruba  South America     29.17
3           4           Senegal         Africa     28.90
4           5        Mauritania         Africa     28.82


In [15]:
# define the URL of the webpage and download the raw HTML
url = "https://en.wikipedia.org/wiki/List_of_countries_by_employment_rate"

# modern websites often require headers set correctly before 
# we can remotely read the website contents (HTML)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}


data_Table_1 = pd.read_html(url, storage_options=headers)
employment_df = data_Table_1[0]
employment_df

,Country,Rate (%),Year,Source
0,Iceland,86.2,Q3 2025,[5]
1,Netherlands,82.4,Q3 2025,[5]
2,Malta *,81.3,Q3 2025,[5]
3,Switzerland,79.8,Q3 2025,[5]
4,Japan,79.3,Q2 2024,[6]
...,...,...,...,...
65,Moldova,47.6,Q2 2021,[10]
66,Egypt,47.0,2024,ILO[8]
67,South Africa *,40.8,2024,OECD[9]
68,Kosovo,39.0,2024,ILO[8]


In [16]:
# define the URL of the webpage and download the raw HTML
url = "https://en.wikipedia.org/wiki/List_of_countries_by_unemployment_rate"

# modern websites often require headers set correctly before 
# we can remotely read the website contents (HTML)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}


data_Table_2 = pd.read_html(url, storage_options=headers)
unemployment_df = data_Table_2[0]
unemployment_df.head()

Country/Territory Unemployment rate (%)                             \
  Country/Territory         CIA[5] (2024) WB[6] (2024) IMF[7] (2026)   
0     Afghanistan *                  13.3         13.3             –   
1         Albania *                  10.3         10.3           8.7   
2         Algeria *                  11.5         11.4             –   
3           Andorra                     –            –           1.1   
4          Angola *                  14.5         14.5             –   

                
  GE[8] (2024)  
0        13.30  
1        10.25  
2        11.43  
3            –  
4        14.46

In [17]:
# Clean the Employment Table 
employment_df = employment_df.rename(columns = {
    "Country" : "Country",
    "Rate (%)" : "EmploymentRate"
})
# Extract the year from the values
employment_df['Year'] = (employment_df['Year'].str.extract(r"(\d{4})").astype(int))
employment_df.head()

,Country,EmploymentRate,Year,Source
0,Iceland,86.2,2025,[5]
1,Netherlands,82.4,2025,[5]
2,Malta *,81.3,2025,[5]
3,Switzerland,79.8,2025,[5]
4,Japan,79.3,2024,[6]


In [18]:
# Keep only 2020 or newer
employment_df = employment_df[employment_df["Year"] >= 2020]

# Filterout the columns we actually need
employment_df = employment_df[["Country", "EmploymentRate"]]
employment_df

,Country,EmploymentRate
0,Iceland,86.2
1,Netherlands,82.4
2,Malta *,81.3
3,Switzerland,79.8
4,Japan,79.3
...,...,...
65,Moldova,47.6
66,Egypt,47.0
67,South Africa *,40.8
68,Kosovo,39.0


In [19]:
#See the columns of Table 2
unemployment_df.columns

MultiIndex([(    'Country/Territory', 'Country/Territory'),
            ('Unemployment rate (%)',     'CIA[5] (2024)'),
            ('Unemployment rate (%)',      'WB[6] (2024)'),
            ('Unemployment rate (%)',     'IMF[7] (2026)'),
            ('Unemployment rate (%)',      'GE[8] (2024)')],
           )

In [20]:
# Prepare the table 2
# Select Country and World Bank unemployment rate
unemployment_df = unemployment_df[
    [
        ("Country/Territory", "Country/Territory"),
        ("Unemployment rate (%)", "WB[6] (2024)")
    ]
]

# Give them simple names
unemployment_df.columns = ["Country", "UnemploymentRate"]

unemployment_df.head()


,Country,UnemploymentRate
0,Afghanistan *,13.3
1,Albania *,10.3
2,Algeria *,11.4
3,Andorra,–
4,Angola *,14.5


In [21]:
# Combine the two tables
combined_df = pd.merge(
    employment_df,
    unemployment_df,
    on="Country",
    how="inner"
)

combined_df.head()

,Country,EmploymentRate,UnemploymentRate
0,Malta *,81.3,2.7
1,New Zealand *,78.6,4.9
2,Sweden *,77.4,8.5
3,Australia *,77.1,4.1
4,Denmark *,77.1,5.6


In [22]:
# Clean the County column
combined_df["Country"] = (
    combined_df["Country"]
    .str.replace("*", "", regex=False)
    .str.strip()
)
combined_df.head()

,Country,EmploymentRate,UnemploymentRate
0,Malta,81.3,2.7
1,New Zealand,78.6,4.9
2,Sweden,77.4,8.5
3,Australia,77.1,4.1
4,Denmark,77.1,5.6


In [23]:
# Load the OECD-Table
oecd_table = data_Table_2[1]
oecd_table.head()

,Country,Total,15–24 year-olds,25–70 year-olds
0,Colombia *,10.20,21.60,8.40
1,Spain *,9.93,23.01,8.93
2,Greece *,9.60,23.70,8.70
3,Chile *,8.80,21.60,7.80
4,Turkey *,8.50,16.30,7.10


In [24]:
# Clean the country names
oecd_table["Country"] = (
    oecd_table["Country"]
    .str.replace("*", "", regex=False)
    .str.strip()
)
# Rename columns
oecd_df = oecd_table.rename(columns={
    "15–24 year-olds": "EmploymentRate15_24",
    "25–70 year-olds": "EmploymentRate25_70"
})
# Keep only required columns
oecd_df = oecd_df[
    ["Country", "EmploymentRate15_24", "EmploymentRate25_70"]
]
# Merge
combined_df = pd.merge(
    combined_df,
    oecd_df,
    on="Country",
    how="left"
)
combined_df.head()

,Country,EmploymentRate,UnemploymentRate,EmploymentRate15_24,EmploymentRate25_70
0,Malta,81.3,2.7,NaN,NaN
1,New Zealand,78.6,4.9,14.1,3.2
2,Sweden,77.4,8.5,21.7,6.1
3,Australia,77.1,4.1,9.6,2.7
4,Denmark,77.1,5.6,14.0,3.9


In [25]:
# Load coffee dataset
coffee_df = pd.read_csv('coffee_sales.csv')
coffee_df.head()

,Date,Sales,Target_sales,Total_expenses
0,1.10.2012,122,90,76
1,1.10.2012,123,90,45
2,2.10.2012,107,90,36
3,3.10.2012,94,100,21
4,4.10.2012,182,80,54


In [26]:
#load temerarture fixed dataset
temp_fixed_df = pd.read_csv('temperatures_fixed.csv')
temp_fixed_df.head()


,Date,Temperature (C)
0,1.1.2006,4.075000
1,1.1.2007,3.806713
2,1.1.2008,-5.663194
3,1.1.2009,-4.850926
4,1.1.2010,7.807407


In [27]:
# Check the both date columns are in same datatype
print(coffee_df['Date'].dtypes)
print(temp_fixed_df['Date'].dtypes)

object
object


In [28]:
# Merge the two dataset
merged_fixed = pd.merge(coffee_df, temp_fixed_df, on='Date', how='left')
print(merged_fixed.head())

        Date  Sales  Target_sales  Total_expenses  Temperature (C)
0  1.10.2012    122            90              76        19.822917
1  1.10.2012    123            90              45        19.822917
2  2.10.2012    107            90              36        19.020602
3  3.10.2012     94           100              21        15.820139
4  4.10.2012    182            80              54        15.110648


In [29]:
# Load temperature_unfixed dataset
temp_unfixed_df = pd.read_csv('temperatures_unfixed.csv')
temp_unfixed_df.head()

,Summary,Temperature (C),Date,Hour
0,Partly Cloudy,9.472222,31.3.2006,22
1,Partly Cloudy,9.355556,31.3.2006,23
2,Mostly Cloudy,9.377778,1.4.2006,0
3,Partly Cloudy,8.288889,1.4.2006,1
4,Partly Cloudy,9.222222,1.4.2006,3


In [30]:
# Drop Summery and Hour
# Then group by Date and Average
temp_unfixed_avg = temp_unfixed_df.drop(columns=['Summary', 'Hour']).groupby('Date').mean().reset_index()

# Then we can merge
merged_unfixed = pd.merge(coffee_df, temp_unfixed_avg, on='Date', how='left')
print(merged_unfixed.head())

        Date  Sales  Target_sales  Total_expenses  Temperature (C)
0  1.10.2012    122            90              76        19.822917
1  1.10.2012    123            90              45        19.822917
2  2.10.2012    107            90              36        19.020602
3  3.10.2012     94           100              21        15.820139
4  4.10.2012    182            80              54        15.110648


In [31]:
# Summary is text.So we can't average it
# Instead we pick the most frequent (mode) summary for each day:
# Most common weather description perd day
# Get from Chatgpt
summary_per_day = temp_unfixed_df.groupby('Date')['Summary'].agg(lambda x : x.mode()[0]).reset_index()

# combine daily temperature + daily summary into one small table
daily_weather = pd.merge(temp_unfixed_avg, summary_per_day, on='Date', how='left')

# merge with coffee sales
merged_unfixed_full = pd.merge(coffee_df, daily_weather, on='Date', how='left')
print(merged_unfixed_full.head())

        Date  Sales  Target_sales  Total_expenses  Temperature (C)  \
0  1.10.2012    122            90              76        19.822917   
1  1.10.2012    123            90              45        19.822917   
2  2.10.2012    107            90              36        19.020602   
3  3.10.2012     94           100              21        15.820139   
4  4.10.2012    182            80              54        15.110648   

         Summary  
0  Mostly Cloudy  
1  Mostly Cloudy  
2  Mostly Cloudy  
3  Partly Cloudy  
4  Partly Cloudy  


In [33]:
# load the raw JSON file into a Python list of dictionaries
with open('complex.json') as f:
    data = json.load(f)

# flatten the nested dictionaries into columns
df = pd.json_normalize(data)
print(df.head())

   id      brand          model  year  price           ownership.company  \
0   1       Ford           F250  1997  72213              Johns and Sons   
1   2      Dodge  Grand Caravan  1999  42370    Crist, Hyatt and Leannon   
2   3       Audi             A5  2011  19191              Bosco and Sons   
3   4    Hyundai       Veracruz  2012  21956                   Haley Inc   
4   5  Chevrolet    TrailBlazer  2009  53066  Zulauf, Nolan and Franecki   

  ownership.acquired ownership.payment_info.credit_card  \
0         24/10/2017                                jcb   
1           7/6/2020                       instapayment   
2         23/11/2018                                jcb   
3           5/6/2016                           bankcard   
4          14/1/2021                                jcb   

          ownership.payment_info.iban  
0       SA47 315V KYTA 8FVT NCGN MM7G  
1   FR74 8157 1767 53I2 VRAN ZMIB U11  
2  LB32 4989 VCVZ S5BZ X2CT JKG3 QRAW  
3                 BE62 4961 

In [34]:
# load the file using the built-in json module
with open('simple.json') as f:
    data = json.load(f)

print(data[0])

{'id': 1, 'brand': 'Chrysler', 'model': 'Voyager', 'year': 2001, 'owned_by': 'Wolff-Trantow', 'acquired': '28/12/2016', 'price': 11565}


In [35]:
# Calculate the avearge price manually
total_price = 0
count = 0

for car in data:
    total_price = total_price + car['price']
    count = count + 1

average_price = total_price / count

print(f'Average Price:',average_price)


Average Price: 42373.86


In [36]:
import xml.etree.ElementTree as ET

# get the  fields the easy way
df = pd.read_xml('complex.xml')
df.head()

,id,brand,model,year,ownership,price
0,1,Mazda,B-Series Plus,1994,NaN,69221
1,2,Ford,F350,2006,NaN,13859
2,3,BMW,7 Series,2012,NaN,78125
3,4,Subaru,Baja,2004,NaN,32941
4,5,Audi,S6,2007,NaN,46761


In [ ]:
# Parse the raw XML tree to pull out the nested ownership data
# Refered an example obtained from Chatgpt
tree = ET.parse('complex.xml')
root = tree.getroot()

ownership_rows = []
for car in root.findall('car'):
    ownership_rows.append({
        'id': int(car.find('id').text),
        'company': car.find('ownership/company').text,
        'acquired': car.find('ownership/acquired').text,
        'credit_card': car.find('ownership/payment_info/credit_card').text,
        'iban': car.find('ownership/payment_info/iban').text
    })

ownership_df = pd.DataFrame(ownership_rows)

# drop the useless empty column, then merge in the real ownership data
df = df.drop(columns=['ownership'])
final_df = pd.merge(df, ownership_df, on='id', how='left')

print(final_df.head())

   id   brand          model  year  price                            company  \
0   1   Mazda  B-Series Plus  1994  69221                  Mueller-VonRueden   
1   2    Ford           F350  2006  13859           Simonis, Graham and Veum   
2   3     BMW       7 Series  2012  78125                         Hammes LLC   
3   4  Subaru           Baja  2004  32941  MacGyver, Oberbrunner and McGlynn   
4   5    Audi             S6  2007  46761          Hand, Johnston and Hickle   

     acquired          credit_card                                iban  
0   25/3/2020                  jcb            LU43 964Q PJUD Z9WI JVSR  
1    6/4/2020  diners-club-enroute        AE82 5889 6103 2911 2983 729  
2  25/12/2017                  jcb  AZ52 PNUH TWPT YQY9 RMLE BMXO 4JIT  
3    2/6/2019           mastercard   FR22 5863 0754 268V RGIQ GHX5 Y27  
4  12/10/2016                  jcb            LT37 9834 6062 9512 0523  


In [38]:
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://www.toptal.com/developers/python/web-scraping-with-python")

print(driver.title)

driver.quit()

Web Scraping Using Python Selenium | Toptal®
